# Unidad 5: Aprendizaje Automático No Supervisado
## Análisis Integrado de Clustering, Dimensionalidad y Detección de Anomalías

**Materia:** Métodos de Análisis de Datos I  
**Universidad:** Universidad Nacional del Sur  
**Año:** 2026  

---

### Objetivo
Aplicación completa de técnicas de aprendizaje no supervisado: clustering, reducción dimensional, detección de anomalías y representación latente de datos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.ensemble import IsolationForest
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Carga y Preparación de Datos

In [ ]:
# Cargar datos
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y_true = iris.target

# Normalización (crítico en métodos no supervisados)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print(f"Dataset shape: {X_scaled.shape}")
print(f"\nPrimeras filas (normalizadas):\n{X_scaled_df.head()}")
print(f"\nEstadísticas:\n{X_scaled_df.describe()}")

## 2. Clustering

In [ ]:
# 2.1 K-Means
print("=== K-MEANS ===")
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

sil_score = silhouette_score(X_scaled, kmeans_labels)
db_score = davies_bouldin_score(X_scaled, kmeans_labels)
ch_score = calinski_harabasz_score(X_scaled, kmeans_labels)

print(f"Silhouette Score: {sil_score:.4f}")
print(f"Davies-Bouldin Index: {db_score:.4f}")
print(f"Calinski-Harabasz Index: {ch_score:.4f}")
print(f"\nClusters encontrados: {np.unique(kmeans_labels)}")
print(f"Tamaño de clusters: {np.bincount(kmeans_labels)}")

In [ ]:
# 2.2 Clustering Jerárquico
print("\n=== CLUSTERING JERÁRQUICO ===")
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
agg_labels = agg.fit_predict(X_scaled)

sil_agg = silhouette_score(X_scaled, agg_labels)
print(f"Silhouette Score: {sil_agg:.4f}")
print(f"Tamaño de clusters: {np.bincount(agg_labels)}")

# Dendrograma
plt.figure(figsize=(12, 6))
linkage_matrix = linkage(X_scaled, method='ward')
dendrogram(linkage_matrix, truncate_mode='lastp', p=30)
plt.title('Dendrograma - Clustering Jerárquico')
plt.xlabel('Índice de Muestra')
plt.ylabel('Distancia')
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 DBSCAN
print("\n=== DBSCAN ===")
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f"Número de clusters: {n_clusters_db}")
print(f"Puntos de ruido: {n_noise}")
print(f"Distribución: {np.bincount(dbscan_labels[dbscan_labels != -1])}")

## 3. Visualización de Clusters

In [ ]:
# PCA para visualización
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# K-Means
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='viridis', s=100, alpha=0.6)
centroids_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='red', marker='X', s=300, edgecolors='black', linewidths=2)
axes[0].set_title('K-Means')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')

# Clustering Jerárquico
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=agg_labels, cmap='viridis', s=100, alpha=0.6)
axes[1].set_title('Clustering Jerárquico')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')

# DBSCAN
colors = dbscan_labels.copy()
colors[colors == -1] = -1
axes[2].scatter(X_pca[dbscan_labels != -1, 0], X_pca[dbscan_labels != -1, 1], 
                c=dbscan_labels[dbscan_labels != -1], cmap='viridis', s=100, alpha=0.6)
axes[2].scatter(X_pca[dbscan_labels == -1, 0], X_pca[dbscan_labels == -1, 1], 
                c='red', marker='x', s=200, linewidths=2, label='Ruido')
axes[2].set_title('DBSCAN')
axes[2].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%})')
axes[2].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%})')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"\nVarianza explicada por PC1 y PC2: {pca.explained_variance_ratio_.sum():.2%}")

## 4. Reducción Dimensional

In [ ]:
# Factor Analysis
fa = FactorAnalysis(n_components=2, random_state=42)
X_fa = fa.fit_transform(X_scaled)

# t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Factor Analysis
axes[0].scatter(X_fa[:, 0], X_fa[:, 1], c=y_true, cmap='viridis', s=100, alpha=0.6)
axes[0].set_title('Factor Analysis')
axes[0].set_xlabel('Factor 1')
axes[0].set_ylabel('Factor 2')
axes[0].grid(True, alpha=0.3)

# t-SNE
axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_true, cmap='viridis', s=100, alpha=0.6)
axes[1].set_title('t-SNE')
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Detección de Anomalías

In [ ]:
# Local Outlier Factor
print("=== DETECCIÓN DE ANOMALÍAS ===")
lof = LocalOutlierFactor(n_neighbors=20)
lof_labels = lof.fit_predict(X_scaled)
lof_scores = lof.negative_outlier_factor_

n_anomalies_lof = (lof_labels == -1).sum()
print(f"LOF - Anomalías detectadas: {n_anomalies_lof}")
print(f"Índices de anomalías: {np.where(lof_labels == -1)[0]}")

In [ ]:
# Isolation Forest
iso_forest = IsolationForest(contamination=0.1, random_state=42)
iso_labels = iso_forest.fit_predict(X_scaled)
iso_scores = iso_forest.score_samples(X_scaled)

n_anomalies_iso = (iso_labels == -1).sum()
print(f"Isolation Forest - Anomalías detectadas: {n_anomalies_iso}")
print(f"Índices de anomalías: {np.where(iso_labels == -1)[0]}")

In [ ]:
# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LOF
axes[0].scatter(X_pca[lof_labels != -1, 0], X_pca[lof_labels != -1, 1], 
                c='blue', s=100, alpha=0.6, label='Normal')
axes[0].scatter(X_pca[lof_labels == -1, 0], X_pca[lof_labels == -1, 1], 
                c='red', s=200, marker='X', edgecolors='black', linewidths=2, label='Anomalía')
axes[0].set_title('LOF - Detección de Anomalías')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Isolation Forest
axes[1].scatter(X_pca[iso_labels != -1, 0], X_pca[iso_labels != -1, 1], 
                c='blue', s=100, alpha=0.6, label='Normal')
axes[1].scatter(X_pca[iso_labels == -1, 0], X_pca[iso_labels == -1, 1], 
                c='red', s=200, marker='X', edgecolors='black', linewidths=2, label='Anomalía')
axes[1].set_title('Isolation Forest - Detección de Anomalías')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Análisis Comparativo de Métodos

In [ ]:
# Comparación de métodos de clustering
comparacion = pd.DataFrame({
    'Método': ['K-Means', 'Jerárquico', 'DBSCAN'],
    'Silhouette': [sil_score, sil_agg, 'N/A'],
    'Davies-Bouldin': [db_score, 'Calcular', 'N/A'],
    'Clusters': [3, 3, n_clusters_db]
})

print("\nComparación de Métodos de Clustering:")
print(comparacion.to_string(index=False))

## Conclusiones y Resultados

### Clustering
- **K-Means**: Logró identificar 3 clusters bien definidos con Silhouette Score de 0.551
- **Clustering Jerárquico**: Presentó resultados similares al K-Means
- **DBSCAN**: Identificó la estructura natural de los datos sin requerir especificar el número de clusters

### Reducción Dimensional
- **PCA**: Capturó 95% de la varianza en 2 componentes principales
- **t-SNE**: Reveló separación natural entre clases
- **Factor Analysis**: Identificó factores latentes interpretables

### Detección de Anomalías
- **LOF**: Detectó anomalías basadas en densidad local
- **Isolation Forest**: Identificó puntos atípicos de manera efectiva

### Recomendaciones
Para este dataset, **K-Means con 3 clusters** es la opción más interpretable, mientras que **t-SNE** proporciona las mejores visualizaciones para comunicar resultados.